# Training and preparing the models

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
import torchvision.transforms as transforms
import torchvision.datasets as datasets
import torchvision.models as models
import torch.optim as optim  # Added missing import
from PIL import Image
import requests
from io import BytesIO
import matplotlib.pyplot as plt  # For visualization
import torchvision

In [ ]:
# function for model
# Function for model
def get_model(model_name, num_classes, pretrained=True, aux_logits=True):
    """
    Returns the specified pre-trained model from torchvision.

    Args:
        model_name (str): Name of the model ('resnet50', 'inception_v3', 'vgg16').
        num_classes (int): Number of classes in the dataset.
        pretrained (bool): Whether to use pre-trained weights. Default is True.

    Returns:
        torch.nn.Module: The specified model.
    """
    model_name = model_name.lower()  # Ensure the model name is case-insensitive

    if model_name == 'resnet50':
        model = models.resnet50(pretrained=pretrained)
        model.fc = nn.Linear(model.fc.in_features, num_classes)
    elif model_name == 'inception_v3':
        model = models.inception_v3(pretrained=pretrained, aux_logits=aux_logits)
        model.fc = nn.Linear(model.fc.in_features, num_classes)
    elif model_name == 'vgg16':
        model = models.vgg16(pretrained=pretrained)
        model.classifier[6] = nn.Linear(4096, num_classes)
    else:
        raise ValueError(f"Unsupported model name '{model_name}'. Choose from 'resnet50', 'inception_v3', or 'vgg16'.")

    return model

# function for dataset
# Function for dataset
def get_dataset(dataset_name, image_size=224):
    """
    Load the specified dataset with separate transformations for training and testing.

    Args:
        dataset_name (str): The name of the dataset ('cifar10' or 'imagenet').
        image_size (int): The size to which images will be resized.

    Returns:
        DataLoader: Train and test loaders for the dataset.
        int: Number of classes in the dataset.
    """
    if dataset_name == 'cifar10':
        # Training transforms with data augmentation
        train_transform = transforms.Compose([
            transforms.RandomHorizontalFlip(),
            transforms.RandomCrop(32, padding=4),
            transforms.Resize(image_size),
            transforms.ToTensor(),
            transforms.Normalize((0.4914, 0.4822, 0.4465), 
                                 (0.2023, 0.1994, 0.2010)),
        ])

        # Testing transforms without data augmentation
        test_transform = transforms.Compose([
            transforms.Resize(image_size),
            transforms.ToTensor(),
            transforms.Normalize((0.4914, 0.4822, 0.4465), 
                                 (0.2023, 0.1994, 0.2010)),
        ])

        train_dataset = datasets.CIFAR10(root='./data', train=True, download=True, transform=train_transform)
        test_dataset = datasets.CIFAR10(root='./data', train=False, download=True, transform=test_transform)

        num_classes = 10

    elif dataset_name == 'imagenet':
        # Training transforms with data augmentation
        train_transform = transforms.Compose([
            transforms.RandomResizedCrop(image_size),
            transforms.RandomHorizontalFlip(),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406],
                                 std=[0.229, 0.224, 0.225]),
        ])

        # Testing transforms without data augmentation
        test_transform = transforms.Compose([
            transforms.Resize(image_size),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406],
                                 std=[0.229, 0.224, 0.225]),
        ])

        # Replace this with actual ImageNet data paths
        train_dataset = datasets.ImageFolder(root='./data/imagenet/train', transform=train_transform)
        test_dataset = datasets.ImageFolder(root='./data/imagenet/val', transform=test_transform)

        num_classes = 1000

    else:
        raise ValueError("Unsupported dataset. Choose from 'cifar10' or 'imagenet'.")

    train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True, num_workers=4)
    test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False, num_workers=4)

    return train_loader, test_loader, num_classes


def load_trained_model(model_name, dataset_name, num_classes, device):
    """
    Loads the trained model with saved weights.

    Args:
        model_name (str): The name of the model ('resnet50', 'inception_v3', 'vgg16').
        dataset_name (str): The dataset name ('cifar10' or 'imagenet').
        num_classes (int): Number of classes in the dataset.
        device (torch.device): The device to load the model onto.

    Returns:
        torch.nn.Module: The trained model loaded with saved weights.
    """
    # Initialize the model with aux_logits=False for evaluation
    model = get_model(model_name, num_classes, pretrained=True, aux_logits=True).to(device)
    model_path = f"best_{dataset_name}_{model_name}_model.pth"
    print("Loading ", model_path)
    try:
        model.load_state_dict(torch.load(model_path, map_location=device))
        print(f"Loaded model weights from '{model_path}' successfully.")
    except FileNotFoundError:
        print(f"Model weights file '{model_path}' not found.")
        raise
    model.eval()  # Set the model to evaluation mode
    return model




# fnction for loading the model and partitioning 

In [ ]:
class PartitionedResNet50(nn.Module):
    def __init__(self, full_model, device):
        super(PartitionedResNet50, self).__init__()

        # Assign device
        self.device = device

        # Partition 1: Initial convolutional layer and first block
        self.device1 = nn.Sequential(
            full_model.conv1,
            full_model.bn1,
            full_model.relu,
            full_model.maxpool,
            full_model.layer1,  # First residual block
        ).to(self.device)

        # Partition 2: Second block
        self.device2 = nn.Sequential(
            full_model.layer2,  # Second residual block
        ).to(self.device)

        # Partition 3: Third block
        self.device3 = nn.Sequential(
            full_model.layer3,  # Third residual block
        ).to(self.device)

        # Partition 4: Fourth block and final layers
        self.device4 = nn.Sequential(
            full_model.layer4,  # Fourth residual block
            nn.AdaptiveAvgPool2d((1, 1)),
            nn.Flatten(),
            full_model.fc,
        ).to(self.device)

    def forward(self, x):
        """
        Forward pass through the partitioned ResNet50 model.

        Args:
            x (torch.Tensor): Input tensor.

        Returns:
            torch.Tensor: Final output logits.
        """
        # Ensure input is on the correct device
        x = x.to(self.device)

        # Forward pass through partitions
        x = self.device1(x)
        x = self.device2(x)
        x = self.device3(x)
        x = self.device4(x)

        return x  # Return the final output logits

In [ ]:
# Evaluate the full model to confirm ~90% accuracy
def evaluate_full_model(full_model, test_loader, device):
    full_model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for batch_idx, (images, labels) in enumerate(test_loader):
            images = images.to(device)
            labels = labels.to(device)

            outputs = full_model(images)
            logits = outputs.logits if hasattr(outputs, 'logits') else outputs

            _, predicted = torch.max(logits, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

            # (Optional) Print progress
            if (batch_idx + 1) % 10 == 0 or (batch_idx + 1) == len(test_loader):
                print(f"Full Model: Processed {total} samples")

    accuracy = 100 * correct / total
    print(f"Full Model Test Accuracy: {accuracy:.2f}%")
    return accuracy

def evaluate_partitioned_model(partitioned_model, test_loader, device):
    partitioned_model.eval()  # Ensure the model is in evaluation mode
    correct = 0
    total = 0

    with torch.no_grad():
        for batch_idx, (images, labels) in enumerate(test_loader):
            images = images.to(device)
            labels = labels.to(device)

            outputs = partitioned_model(images)  # Forward pass
            _, predicted = torch.max(outputs, 1)  # Get predictions

            total += labels.size(0)
            correct += (predicted == labels).sum().item()

            # (Optional) Print progress
            if (batch_idx + 1) % 10 == 0 or (batch_idx + 1) == len(test_loader):
                print(f"Processed {total} samples")

    accuracy = 100 * correct / total
    print(f"Partitioned Model Test Accuracy: {accuracy:.2f}%")
    return accuracy

# Function to compare outputs between full and partitioned models
def compare_models(full_model, partitioned_model, test_loader, device, atol=1e-4):
    """
    Compares the outputs of the full model and the partitioned model.

    Args:
        full_model (nn.Module): The full Inception_v3 model.
        partitioned_model (nn.Module): The partitioned Inception_v3 model.
        test_loader (DataLoader): DataLoader for the test dataset.
        device (torch.device): The device to perform computations on.
        atol (float): Absolute tolerance for comparison.

    Returns:
        None
    """
    full_model.eval()
    partitioned_model.eval()

    with torch.no_grad():
        for batch_idx, (images, labels) in enumerate(test_loader):
            images = images.to(device)
            labels = labels.to(device)

            # Full model prediction
            outputs_full = full_model(images)
            logits_full = outputs_full.logits if hasattr(outputs_full, 'logits') else outputs_full

            # Partitioned model prediction
            outputs_partitioned = partitioned_model(images)

            # Compare outputs
            if not torch.allclose(logits_full, outputs_partitioned, atol=atol):
                max_diff = (logits_full - outputs_partitioned).abs().max().item()
                print("Outputs differ between full model and partitioned model.")
                print(f"Max difference: {max_diff}")
            else:
                print("Outputs match between full model and partitioned model.")

            # Process only the first batch for comparison
            break



## Main Execution Block


In [ ]:

# Configuration
model_name = 'resnet50'  # Update to 'resnet50'
dataset_name = 'cifar10'  # Options: 'cifar10', 'imagenet'
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

# Determine image size based on model
image_size = 224  # ResNet50 uses 224x224 images

# Load dataset
train_loader, test_loader, num_classes = get_dataset(dataset_name, image_size=image_size)

# Load the trained full model
full_model = load_trained_model(model_name, dataset_name, num_classes, device)

# Evaluate the full model
full_accuracy = evaluate_full_model(full_model, test_loader, device)

# Create the partitioned model
partitioned_model = PartitionedResNet50(full_model, device)
partitioned_model.eval()

# Evaluate the partitioned model
partitioned_accuracy = evaluate_partitioned_model(partitioned_model, test_loader, device)

# Compare outputs between full and partitioned models
compare_models(full_model, partitioned_model, test_loader, device)